# DNA Sequence GC Content Analysis
This notebook calculates the GC content of a DNA sequence and creates a visualization of the GC content distribution.

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import random
from PIL import Image

# Read data from Excel file
df = pd.read_excel("C:/Users/soulp/Onedrive/Desktop/Figure_16.xlsx")

# Calculate the total reads
total_reads = df['Reads'].sum()

# Calculate the percentage of reads for each bacterial phylum
phylum_percentage = df.groupby('Bacterial Phylum')['Reads'].sum() / total_reads * 100
phylum_percentage = phylum_percentage.reset_index()

# Sort bacterial phyla by percentage in descending order
phylum_percentage = phylum_percentage.sort_values(by='Reads', ascending=False)

# Extract the unique labels from the columns
locations = df['Location'].unique().tolist()
events = df['Event'].unique().tolist()
# Add percentage values to bacterial phyla labels
bacterial_phyla = [f"{row['Bacterial Phylum']} ({row['Reads']:.2f}%)" for _, row in phylum_percentage.iterrows()]

# Create labels list for Sankey diagram, replacing specific location names with empty strings
labels = []
locations_to_hide = ['Sangam']
for loc in locations:
    if loc in locations_to_hide:
        labels.append("")  # Empty string for hidden locations
    else:
        labels.append(loc)
labels.extend(events)
labels.extend(bacterial_phyla)

# Define a function to get the index of a label
def get_index(label):
    if label in locations_to_hide:
        # Get the index of the empty string that corresponds to this location
        return locations_to_hide.index(label)
    return labels.index(label)

# Create the source and target lists
source = []
target = []
value = []

# Add data for the location -> event connections
for location in locations:
    for event in events:
        mask = (df['Location'] == location) & (df['Event'] == event)
        if mask.any():
            source.append(get_index(location))
            target.append(get_index(event))
            value.append(df.loc[mask, 'Reads'].sum())

# Add data for the event -> bacterial phylum connections
for event in events:
    for _, row in phylum_percentage.iterrows():
        phylum = row['Bacterial Phylum']
        phylum_label = f"{phylum} ({row['Reads']:.2f}%)"
        mask = (df['Event'] == event) & (df['Bacterial Phylum'] == phylum)
        if mask.any():
            source.append(get_index(event))
            target.append(get_index(phylum_label))
            value.append(df.loc[mask, 'Reads'].sum())

# Define node colors using an extended and more academic palette
node_colors = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f',
    '#bcbd22', '#17becf', '#aec7e8', '#ffbb78', '#98df8a', '#ff9896', '#c5b0d5', '#c49c94',
    '#f7b6d2', '#c7c7c7', '#dbdb8d', '#9edae5', '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728',
    '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf', '#aec7e8', '#ffbb78',
    '#98df8a', '#ff9896', '#c5b0d5', '#c49c94', '#f7b6d2', '#c7c7c7', '#dbdb8d', '#9edae5'
]

# Assign colors to nodes
node_colors = [node_colors[i % len(node_colors)] for i in range(len(labels))]

# Define gradient link colors
link_colors = []
for i in range(len(source)):
    link_colors.append(f'rgba({random.randint(0,255)}, {random.randint(0,255)}, {random.randint(0,255)}, 0.5)')

# Create the Sankey diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels,  # Include labels for all nodes
        color=node_colors
    ),
    link=dict(
        source=source,
        target=target,
        value=value,
        color=link_colors
    )
)])

fig.update_layout(
    font=dict(family='Times New Roman', size=16, color='black'),
    paper_bgcolor='rgba(0,0,0,0)',  # Transparent background
    plot_bgcolor='rgba(0,0,0,0)',   # Transparent plot background
    height=1100,
    width=1300
)

# Save the figure as a PNG file with a high scale to ensure high DPI
fig.write_image("C:/Users/soulp/Onedrive/Desktop/Figure_16.png", format='png', scale=3)

# Convert the PNG file to a JPEG file
with Image.open("C:/Users/soulp/Onedrive/Desktop/Figure_16.png") as img:
    img = img.convert('RGB')
    img.save("C:/Users/soulp/Onedrive/Desktop/Figure_16.jpg", format='JPEG', quality=95, dpi=(1200, 1200))

fig.show()